--- Day 11: Reactor ---
You hear some loud beeping coming from a hatch in the floor of the factory, so you decide to check it out. Inside, you find several large electrical conduits and a ladder.

Climbing down the ladder, you discover the source of the beeping: a large, toroidal reactor which powers the factory above. Some Elves here are hurriedly running between the reactor and a nearby server rack, apparently trying to fix something.

One of the Elves notices you and rushes over. "It's a good thing you're here! We just installed a new server rack, but we aren't having any luck getting the reactor to communicate with it!" You glance around the room and see a tangle of cables and devices running from the server rack to the reactor. She rushes off, returning a moment later with a list of the devices and their outputs (your puzzle input).

For example:

aaa: you hhh
you: bbb ccc
bbb: ddd eee
ccc: ddd eee fff
ddd: ggg
eee: out
fff: out
ggg: out
hhh: ccc fff iii
iii: out
Each line gives the name of a device followed by a list of the devices to which its outputs are attached. So, bbb: ddd eee means that device bbb has two outputs, one leading to device ddd and the other leading to device eee.

The Elves are pretty sure that the issue isn't due to any specific device, but rather that the issue is triggered by data following some specific path through the devices. Data only ever flows from a device through its outputs; it can't flow backwards.

After dividing up the work, the Elves would like you to focus on the devices starting with the one next to you (an Elf hastily attaches a label which just says you) and ending with the main output to the reactor (which is the device with the label out).

To help the Elves figure out which path is causing the issue, they need you to find every path from you to out.

In this example, these are all of the paths from you to out:

Data could take the connection from you to bbb, then from bbb to ddd, then from ddd to ggg, then from ggg to out.
Data could take the connection to bbb, then to eee, then to out.
Data could go to ccc, then ddd, then ggg, then out.
Data could go to ccc, then eee, then out.
Data could go to ccc, then fff, then out.
In total, there are 5 different paths leading from you to out.

How many different paths lead from you to out?

To begin, get your puzzle input.



So we have nodes with paths. We want to find all paths that flow towards the goal. Presumably, *loops* don't count (unclear from description, or if there are any loops. However, since there are nodes flowing to you, that might be the case)

So we take all paths. If they lead to a node that has already been visited, we don't go further. But that node remembers its upstream nodes (that ultimately came from source). So we explore downwards making sure we hit every path but not to retrace duplicate paths. We make sure that every node remembers its upstream nodes. So that in the end we can create the reverse paths starting at end, following all paths upstream. 

In [45]:
import re
node_dict = {'out' : tuple()}
with open('input.txt') as file:
    for line in file:
        name = re.findall(r'\w{3}', line)[0]
        children = tuple(child for child in re.findall(r'\w{3}', line)[1:])
        node_dict[name] = children
len(node_dict)

636

In [60]:
class Node:

    ledger = set()
    on_path = set()

    def __init__(self, name, parent=None):
        type(self).ledger.add(self)
        self.name = name
        self.parents = set()
        if parent:
            self.parents.add(parent)
        self.collected_children = set() # for counting
        self.multi = 0  # for counting
        self.go_down()
    
    def __repr__(self):
        return f'{self.name} with parents {[p.name for p in self.parents]} and children {[c.name for c in self.children]}'

    @classmethod
    def find_node(cls, name):
        for node in cls.ledger:
            if node.name == name:
                return node
        return False

    def go_down(self):
        self.children = set()
        for name in node_dict[self.name]:
            if child := type(self).find_node(name):
                child.parents.add(self)
                self.children.add(child)
            else:
                child = Node(name, self)
                self.children.add(child)
            

    def go_up(self):
        type(self).on_path.add(self)
        for p in self.parents:
            if p not in type(self).on_path:
                p.go_up()

    def count_paths(self, child=None, multi=1):
        if child:
            self.collected_children.add(child)
        self.multi += multi
        if self.collected_children == self.children & type(self).on_path:
            for p in self.parents:
                if p in type(self).on_path:
                    p.count_paths(self, self.multi)



In [47]:
you = Node('you')
len(Node.ledger)

137

In [48]:
# Now we need to prune the network: Only count connections that ultimately lead to Out. We start at Out and go up through parents
out = Node.find_node('out')
out

out with parents ['bvr', 'lks', 'uhc', 'yjq', 'wwk', 'iaf', 'rbu', 'nhv', 'kjn', 'dxp', 'unp', 'qgl', 'bel', 'isr', 'fcq', 'iqx', 'lhr', 'gqs', 'oca', 'fms', 'fiv', 'hqf', 'led', 'cak', 'uzc', 'dsq'] and children []

In [49]:
out.go_up()
len(Node.leads_to_out)
# We have confirmed that all nodes in Nodes.ledger lead to out. So we count everything.

137

In [50]:
out.count_paths()
you.multi

733

Your puzzle answer was 733.

The first half of this puzzle is complete! It provides one gold star: *

--- Part Two ---
Thanks in part to your analysis, the Elves have figured out a little bit about the issue. They now know that the problematic data path passes through both dac (a digital-to-analog converter) and fft (a device which performs a fast Fourier transform).

They're still not sure which specific path is the problem, and so they now need you to find every path from svr (the server rack) to out. However, the paths you find must all also visit both dac and fft (in any order).

For example:

svr: aaa bbb
aaa: fft
fft: ccc
bbb: tty
tty: ccc
ccc: ddd eee
ddd: hub
hub: fff
eee: dac
dac: fff
fff: ggg hhh
ggg: out
hhh: out
This new list of devices contains many paths from svr to out:

svr,aaa,fft,ccc,ddd,hub,fff,ggg,out
svr,aaa,fft,ccc,ddd,hub,fff,hhh,out
svr,aaa,fft,ccc,eee,dac,fff,ggg,out
svr,aaa,fft,ccc,eee,dac,fff,hhh,out
svr,bbb,tty,ccc,ddd,hub,fff,ggg,out
svr,bbb,tty,ccc,ddd,hub,fff,hhh,out
svr,bbb,tty,ccc,eee,dac,fff,ggg,out
svr,bbb,tty,ccc,eee,dac,fff,hhh,out
However, only 2 paths from svr to out visit both dac and fft.

Find all of the paths that lead from svr to out. How many of those paths visit both dac and fft?

In [53]:
# let's see if fft or dac comes first or if it's mixed:
Node.ledger = set()
fft = Node('fft')
Node.find_node('dac')

dac with parents ['wxq', 'muw', 'sto', 'ywb'] and children ['xor', 'fty', 'iab']

In [54]:
# ok so dac appears downstream of fft. Other way round?
Node.ledger = set()
dac = Node('dac')
Node.find_node('fft')

False

In [64]:
# Knowing that fft is upstream of dac, we can construct our graph in three steps: svr to fft, fft to dac and dac to out. All we need is functionality to prune connections that are not part of our graph. Actually, it's even simpler: Run the algorithm three times to count paths for each leg, then multiply the results

Node.ledger = set()
Node.on_path = set()

svr = Node('svr')
fft = Node.find_node('fft')
fft.go_up()
fft.count_paths()
leg1 = svr.multi

In [65]:
# fft to dac
Node.ledger = set()
Node.on_path = set()

fft = Node('fft')
dac = Node.find_node('dac')
dac.go_up()
dac.count_paths()
leg2 = fft.multi

In [66]:
# dac to out
Node.ledger = set()
Node.on_path = set()

dac = Node('dac')
out = Node.find_node('out')
out.go_up()
out.count_paths()
leg3 = dac.multi

In [67]:
leg1 * leg2 * leg3

290219757077250

That's the right answer! You are one gold star closer to decorating the North Pole.

You have completed Day 11! You can [Share] this victory or [Return to Your Advent Calendar].